# 09 — Mixscape: Escape Detection

**Prerequisites:** Run [07_normalization_dimred.ipynb](07_normalization_dimred.ipynb) first. Requires `data/norman_processed.h5ad`.

**What you'll learn:**
- Why not all cells with a guide show the expected perturbation (escape cells)
- The Mixscape framework: perturbation signature computation + Gaussian mixture model classification
- How to use `pertpy` to run Mixscape
- Interpreting Mixscape output: KO / NP / NT classification
- How escape rate varies across perturbations
- The perturbation UMAP (LPC embedding)
- SCEPTRE: an alternative statistical framework for CRISPR screen analysis

**Estimated time:** 2 hours

In [ ]:
import os
import scanpy as sc
import pertpy as pt
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
import scipy.sparse

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"

adata = sc.read_h5ad(os.path.join(DATA_DIR, "norman_processed.h5ad"))
print(adata)

PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"
ntc_mask = adata.obs["is_ntc"] if "is_ntc" in adata.obs.columns else \
    adata.obs[PERT_COL].str.lower().str.contains("ctrl|control|ntc|non")

ntc_label = adata.obs.loc[ntc_mask, PERT_COL].unique()[0]
print(f"NTC label: '{ntc_label}', cells: {ntc_mask.sum()}")

## 1. Why do escape cells exist?

Not every cell that receives a guide RNA shows the expected transcriptional response. Sources of escape:

| Modality | Main escape mechanism |
|---|---|
| CRISPRko | Indel creates an in-frame mutation that doesn't disrupt function; one allele edited, one not (heterozygous) |
| CRISPRi | dCas9-KRAB recruitment incomplete; promoter not fully accessible; heterochromatin prevents binding |
| CRISPRa | Chromatin state prevents activation; VPR effector not recruited |

In practice, escape rates in CRISPRi/a screens are typically 20–50% of "assigned" cells. Including escape cells in your DE analysis dilutes the true effect size — the mean of perturbed + escape cells moves toward the NTC mean, reducing LFCs and power.

**Mixscape** (Papalexi et al. 2021) separates the two populations.

## 2. Perturbation signature

The perturbation signature is the foundation of Mixscape:

```
For each perturbed cell i:
  signature(i) = expression(i) - mean_NTC_expression
```

This removes the baseline transcriptome and isolates the perturbation-induced deviation. Cells that were not successfully perturbed (escape cells) will have signatures near zero.

In [ ]:
# Compute perturbation signature manually (then verify with pertpy)
import scipy.sparse

X = adata.X
if scipy.sparse.issparse(X):
    X = X.toarray()

ntc_mean = X[ntc_mask.values].mean(axis=0)  # mean NTC expression vector
perturbation_sig = X - ntc_mean  # deviation from NTC for every cell

print(f"Perturbation signature matrix: {perturbation_sig.shape}")
print(f"NTC cells: signature should have mean ≈ 0")
print(f"  NTC signature mean: {perturbation_sig[ntc_mask.values].mean():.4f}")
print(f"  NTC signature std: {perturbation_sig[ntc_mask.values].std():.4f}")

In [ ]:
# Distribution of signature magnitude per perturbation
# Strong perturbations: large magnitude; escape cells: small magnitude

# Focus on single-gene non-NTC cells
single_cells = adata[~adata.obs["is_ntc"] & ~adata.obs.get("is_dual", pd.Series(False, index=adata.obs_names))]
single_X = single_cells.X
if scipy.sparse.issparse(single_X):
    single_X = single_X.toarray()
single_sig = single_X - ntc_mean

# Per-cell signature magnitude (L2 norm)
sig_magnitude = np.linalg.norm(single_sig, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sig_magnitude, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
ax.set_xlabel("Perturbation signature magnitude (L2 norm)")
ax.set_ylabel("Cells")
ax.set_title("Distribution of perturbation signature magnitude\n(single-target cells only)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "09_signature_magnitude.png"), dpi=150, bbox_inches="tight")
plt.show()

print("A bimodal distribution would indicate a mix of perturbed and escape cells.")

## 3. Mixscape with pertpy

In [ ]:
# pertpy Mixscape
# See: https://pertpy.readthedocs.io/en/latest/tutorials/notebooks/mixscape.html

# Mixscape requires:
#  1. The perturbation column in obs
#  2. The NTC label
#  3. A normalized expression matrix in adata.X

adata_ms = adata.copy()

# Ensure adata.X is normalized log1p (not raw counts)
X_sample = adata_ms.X[:5, :5]
if scipy.sparse.issparse(X_sample):
    X_sample = X_sample.toarray()
is_integer = np.allclose(X_sample, np.round(X_sample))
if is_integer:
    print("Warning: adata.X appears to contain raw counts. Normalizing...")
    sc.pp.normalize_total(adata_ms, target_sum=1e4)
    sc.pp.log1p(adata_ms)

try:
    ms = pt.tl.Mixscape()
    
    # Step 1: compute perturbation signature
    ms.perturbation_signature(
        adata_ms,
        pert_key=PERT_COL,
        control=ntc_label,
    )
    print("Perturbation signature computed.")
    print("New layers/obsm:", list(adata_ms.layers.keys()), list(adata_ms.obsm.keys()))
    
    # Step 2: run Mixscape classification
    ms.mixscape(
        adata_ms,
        control=ntc_label,
        pert_key=PERT_COL,
    )
    print("Mixscape classification complete.")
    
    # Check output
    if "mixscape_class" in adata_ms.obs.columns:
        print("\nMixscape class distribution:")
        print(adata_ms.obs["mixscape_class"].value_counts())
    
    MIXSCAPE_SUCCESS = True
    
except Exception as e:
    print(f"pertpy Mixscape raised: {e}")
    print("\nFalling back to manual Gaussian mixture model (section 3b).")
    MIXSCAPE_SUCCESS = False

### 3b. Manual Mixscape implementation (fallback)

This section implements the core Mixscape logic from scratch. It demonstrates the algorithm for learning purposes, independent of the pertpy API.

In [ ]:
def manual_mixscape(adata, pert_col, ntc_label, n_components=2):
    """
    Manual Mixscape implementation.
    
    For each perturbation:
    1. Compute perturbation signature = expression - NTC_mean
    2. Project signature onto top PCs of the perturbation-specific space
    3. Fit Gaussian mixture model (2 components) on the 1D projection
    4. Classify cells as 'KO' or 'NP' based on mixture component assignment
    """
    from sklearn.decomposition import PCA
    
    X = adata.X
    if scipy.sparse.issparse(X):
        X = X.toarray()
    
    ntc_idx = (adata.obs[pert_col] == ntc_label).values
    ntc_mean = X[ntc_idx].mean(axis=0)
    
    mixscape_class = pd.Series("NT", index=adata.obs_names)  # default: non-targeting
    mixscape_prob = pd.Series(0.0, index=adata.obs_names)
    
    perturbations = [p for p in adata.obs[pert_col].unique() if p != ntc_label]
    
    for pert in perturbations:
        pert_idx = (adata.obs[pert_col] == pert).values
        combined_idx = pert_idx | ntc_idx
        
        if pert_idx.sum() < 10:
            mixscape_class[adata.obs_names[pert_idx]] = "KO"  # too few cells to classify
            continue
        
        # Perturbation signature for combined cells
        X_combined = X[combined_idx] - ntc_mean
        
        # PCA on the perturbation signature
        n_pc = min(3, X_combined.shape[0] - 1, X_combined.shape[1] - 1)
        pca = PCA(n_components=n_pc, random_state=42)
        X_pca = pca.fit_transform(X_combined)
        
        # GMM on first PC
        try:
            gmm = GaussianMixture(n_components=2, random_state=42)
            gmm.fit(X_pca[:, :1])
            labels = gmm.predict(X_pca[:, :1])
            probs = gmm.predict_proba(X_pca[:, :1])
            
            # The component with higher mean = KO (perturbed)
            means = gmm.means_.flatten()
            ko_component = np.argmax(np.abs(means))  # component farther from 0
            
            combined_obs_names = adata.obs_names[combined_idx]
            pert_obs_names = adata.obs_names[pert_idx]
            
            for obs_name, label, prob in zip(combined_obs_names, labels, probs[:, ko_component]):
                if obs_name in pert_obs_names:
                    mixscape_class[obs_name] = "KO" if label == ko_component else "NP"
                    mixscape_prob[obs_name] = prob
        except:
            # If GMM fails, mark all as KO
            mixscape_class[adata.obs_names[pert_idx]] = "KO"
    
    return mixscape_class, mixscape_prob


if not MIXSCAPE_SUCCESS:
    print("Running manual Mixscape...")
    # Run on single-target cells only
    adata_single = adata[~adata.obs.get("is_dual", pd.Series(False, index=adata.obs_names))].copy()
    
    X_tmp = adata_single.X
    if scipy.sparse.issparse(X_tmp):
        X_tmp = X_tmp.toarray()
    is_int = np.allclose(X_tmp[:5,:5], np.round(X_tmp[:5,:5]))
    if is_int:
        sc.pp.normalize_total(adata_single, target_sum=1e4)
        sc.pp.log1p(adata_single)
    
    # Use subset for speed
    n_perts = min(30, adata_single.obs[PERT_COL].nunique())
    sample_perts = [ntc_label] + [p for p in adata_single.obs[PERT_COL].unique() if p != ntc_label][:n_perts-1]
    adata_demo = adata_single[adata_single.obs[PERT_COL].isin(sample_perts)].copy()
    
    mc, mp = manual_mixscape(adata_demo, PERT_COL, ntc_label)
    adata_demo.obs["mixscape_class"] = mc.values
    adata_ms = adata_demo
    
    print("Manual Mixscape complete.")
    print(adata_ms.obs["mixscape_class"].value_counts())

## 4. Interpret Mixscape output

In [ ]:
# Escape rate per perturbation
if "mixscape_class" in adata_ms.obs.columns:
    non_ntc = adata_ms.obs[~adata_ms.obs["is_ntc"]]
    
    escape_rate = non_ntc.groupby(PERT_COL)["mixscape_class"].apply(
        lambda x: (x == "NP").mean()
    ).sort_values(ascending=False)
    
    print("Top perturbations by escape rate:")
    print(escape_rate.head(10))
    print(f"\nMean escape rate across all perturbations: {escape_rate.mean():.2%}")
    
    fig, ax = plt.subplots(figsize=(12, 4))
    escape_rate.plot(kind="bar", ax=ax, color="salmon", edgecolor="none")
    ax.set_xlabel("Perturbation")
    ax.set_ylabel("Escape rate (fraction NP cells)")
    ax.set_title("Mixscape escape rate per perturbation")
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.axhline(0.3, color="red", linestyle="--", linewidth=1, label="30% threshold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "09_escape_rate.png"), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Compare UMAP: all assigned vs. Mixscape-filtered (KO only)
if "X_umap" in adata_ms.obsm:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    umap = adata_ms.obsm["X_umap"]
    
    # All cells colored by Mixscape class
    class_colors = {"KO": "#F44336", "NP": "#FF9800", "NT": "#2196F3"}
    for cls, color in class_colors.items():
        mask = adata_ms.obs.get("mixscape_class", pd.Series("NT", index=adata_ms.obs_names)) == cls
        axes[0].scatter(umap[mask, 0], umap[mask, 1], c=color, label=cls,
                       alpha=0.3, s=2, rasterized=True)
    axes[0].set_title("All cells — Mixscape class")
    axes[0].legend(markerscale=5)
    axes[0].set_xlabel("UMAP 1")
    axes[0].set_ylabel("UMAP 2")
    
    # KO cells only
    ko_mask = adata_ms.obs.get("mixscape_class", pd.Series("NT", index=adata_ms.obs_names)).isin(["KO", "NT"])
    axes[1].scatter(umap[~ko_mask, 0], umap[~ko_mask, 1], c="lightgray", alpha=0.1, s=1, rasterized=True, label="NP (excluded)")
    axes[1].scatter(umap[ko_mask, 0], umap[ko_mask, 1], c="#F44336", alpha=0.3, s=2, rasterized=True, label="KO + NT")
    axes[1].set_title("Mixscape-filtered (KO + NT only)")
    axes[1].legend(markerscale=5)
    axes[1].set_xlabel("UMAP 1")
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "09_mixscape_umap.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 5. SCEPTRE: an alternative statistical framework

SCEPTRE (Barry et al. 2023, *Genome Biology*) takes a different approach than pseudobulk + Mixscape:

- **Model**: negative binomial regression on raw single-cell counts (no pseudobulking)
- **Unit of replication**: guide RNA (not cell) — each guide is treated as one replicate
- **Calibration**: uses NTC guides as empirical null to control type I error
- **Handles escape**: explicitly models guide efficiency as a covariate

**When to use SCEPTRE over pseudobulk:**
- When you have very few cells per guide (< 50) — pseudobulk has poor power
- When you need rigorous type I error control (e.g., regulatory element screens)
- For CRISPR screen analysis of non-bulk-compatible phenotypes

SCEPTRE is implemented as an R package (`sceptre`). A Python interface is in development. For this course we point to the R package documentation.

In [ ]:
print("""
SCEPTRE resources:
  Paper: Barry et al. 2023, Genome Biology
  R package: https://github.com/Katsevich-Lab/sceptre
  Tutorial: https://katsevich-lab.github.io/sceptre/

SCEPTRE workflow (R):
  1. Create sceptre_object from single-cell count matrix + guide assignments
  2. Set analysis parameters: NTC guides, discovery pairs (gene x perturbation)
  3. Run calibration check using NTC guides
  4. Run discovery analysis
  5. Interpret results: log2FC, p-value, Benjamini-Hochberg FDR

Key SCEPTRE advantages:
  - Well-calibrated: empirically validated type I error on NTC guides
  - No pseudoreplication problem (uses guide-level variance, not cell-level)
  - Handles overdispersion via negative binomial model
  - Resampling-based null distribution is robust to cell heterogeneity
""")

## 6. Save Mixscape-annotated AnnData

In [ ]:
out_path = os.path.join(DATA_DIR, "norman_mixscape.h5ad")
adata_ms.write_h5ad(out_path)
print(f"Saved to {out_path}")

if "mixscape_class" in adata_ms.obs.columns:
    print("Mixscape class distribution:")
    print(adata_ms.obs["mixscape_class"].value_counts())

## Key takeaways

1. Escape cells (NP: not perturbed) are common — typically 20–50% of cells assigned to a guide in CRISPRi/a experiments
2. Mixscape uses a per-perturbation Gaussian mixture model on the perturbation signature to separate KO from NP cells
3. Removing escape cells before DE analysis increases sensitivity and specificity — LFC estimates move farther from zero
4. High escape rates for a specific perturbation may indicate a chromatin-inaccessible TSS or ineffective guide design
5. SCEPTRE provides rigorous type I error control by using guide-level replication and empirical null calibration — the gold standard for regulatory element screens

**Next:** [10_visualization.ipynb](10_visualization.ipynb)